In [221]:
from models.parameters import Parameter
from typing import List, Tuple, Union, Callable, Dict

def test(a,b=2):
    return a + b

#### NUOVA CLASSE PER IL PARAMETER HANDLER

In [222]:
from types import UnionType
from typing import Iterator
from collections import OrderedDict
import numpy as np

# Assuming 'Parameter' is defined elsewhere
# from parameter_module import Parameter


class ParameterHandler:
    """
    Classe che gestisce un insieme di parametri per un modello.

    Attributes:
        _parameters (OrderedDict[str, Parameter]): Dizionario dei parametri.
        _is_inside_model (bool): Indica se il gestore è stato aggiunto a un modello.
        _cache (Dict[str, Any]): Cache per le proprietà dei parametri.
    """

    def __init__(
        self, parameters: Union["Parameter", List["Parameter"]] = None
    ) -> None:
        """
        Inizializza il gestore dei parametri.

        Args:
            parameters (Union[Parameter, List[Parameter]], opzionale): Un singolo parametro o una lista di parametri.
        """
        self._parameters = OrderedDict()
        self._is_inside_model = (
            False  # Una volta dentro modello non posso aggiungere parametri
        )
        self._cache = {}

        

        if isinstance(parameters, Parameter):
            self.add_parameter(parameters)
        elif isinstance(parameters, list):
            for param in parameters:
                self.add_parameter(param)
        elif parameters is None:
            pass
        else:
            raise TypeError(
                "Parameters must be of type Parameter or List[Parameter]",
                type(parameters),
            )

    def _invalidate_cache(self) -> None:
        """
        Invalida la cache dei parametri.
        """
        self._cache = {}

    @property
    def is_inside_model(self) -> bool:
        """
        Indica se il gestore è stato aggiunto a un modello.

        Returns:
            bool: True se è stato aggiunto a un modello, False altrimenti.
        """
        return self._is_inside_model

    def lock(self) -> None:
        """
        Blocca il gestore per prevenire l'aggiunta di nuovi parametri.
        """
        self._is_inside_model = True

    def unlock(self) -> None:
        """
        Sblocca il gestore per permettere l'aggiunta di nuovi parametri.
        """
        self._is_inside_model = False

    @property
    def parameters_values(self) -> List[float]:
        """
        Ritorna i valori dei parametri.

        Returns:
            List[float]: Lista dei valori dei parametri.
        """

        return [p.value for p in self]

    @property
    def parameters_names(self) -> List[str]:
        """
        Ritorna i nomi dei parametri.

        Returns:
            List[str]: Lista dei nomi dei parametri.
        """
        if "parameters_names" not in self._cache:
            self._cache["parameters_names"] = list(self._parameters.keys())
        return self._cache["parameters_names"]

    @property
    def parameters_bounds(self) -> List[Tuple[float, float]]:
        """
        Ritorna i limiti dei parametri.

        Returns:
            List[Tuple[float, float]]: Lista dei limiti dei parametri.
        """
        return [p.bounds for p in self]

    @property
    def n_free_params(self) -> int:
        """
        Ritorna il numero di parametri liberi.

        Returns:
            int: Numero di parametri liberi.
        """
        return len(self.free_parameters)

    @property
    def free_parameters(self) -> List["Parameter"]:
        """
        Ritorna solo i parametri liberi.

        Returns:
            List[Parameter]: Lista dei parametri liberi.
        """
        if "free_parameters" not in self._cache:
            self._cache["free_parameters"] = [p for p in self if not p.frozen]
        return self._cache["free_parameters"]

    @property
    def frozen_parameters(self) -> List["Parameter"]:
        """
        Ritorna solo i parametri congelati.

        Returns:
            List[Parameter]: Lista dei parametri congelati.
        """
        if "frozen_parameters" not in self._cache:
            self._cache["frozen_parameters"] = [p for p in self if p.frozen]
        return self._cache["frozen_parameters"]
    
    def _map_names_to_indices(self, key: str) -> int:
        """
        Mappa il nome di un parametro al corrispondente indice.

        Args:
            key (str): Nome del parametro.

        Returns:
            int: Indice del parametro.

        Raises:
            KeyError: Se il nome del parametro non è trovato.
        """
        try:
            return self.parameters_names.index(key)
        except ValueError:
            raise KeyError(f"Key '{key}' not found in the dictionary")

    def _map_indices_to_names(self, index: int) -> str:
        """
        Mappa l'indice di un parametro al corrispondente nome.

        Args:
            index (int): Indice del parametro.

        Returns:
            str: Nome del parametro.

        Raises:
            IndexError: Se l'indice è fuori dai limiti.
        """
        keys = list(self._parameters.keys())
        if index < 0 or index >= len(keys):
            raise IndexError(f"Index '{index}' is out of bounds for the dictionary")
        return keys[index]

    def set_values(
        self, values: Union[List, Dict], include_frozen: bool = False
    ) -> None:
        """
        Imposta i valori dei parametri.

        Args:
            values (Union[List, Dict]): Valori da assegnare (lista o dizionario).
            include_frozen (bool, opzionale): Se includere i parametri congelati. Default è False.
        """
        self._assign_attribute(values, "value", include_frozen=include_frozen)
        self._invalidate_cache()

    def set_bounds(
        self, bounds: Union[List, Dict], include_frozen: bool = False
    ) -> None:
        """
        Imposta i limiti dei parametri.

        Args:
            bounds (Union[List, Dict]): Limiti da assegnare (lista o dizionario).
            include_frozen (bool, opzionale): Se includere i parametri congelati. Default è False.
        """
        self._assign_attribute(bounds, "bounds", include_frozen=include_frozen)
        self._invalidate_cache()

    def set_frozen(
        self, is_frozen: Union[List, Dict], include_frozen: bool = False
    ) -> None:
        """
        Imposta lo stato di congelamento dei parametri.

        Args:
            is_frozen (Union[List, Dict]): Stato di congelamento da assegnare (lista o dizionario).
            include_frozen (bool, opzionale): Se includere i parametri già congelati. Default è False.
        """
        self._assign_attribute(is_frozen, "frozen", include_frozen=include_frozen)
        self._invalidate_cache()

    def _assign_attribute(
        self, items: Union[List, Dict], attribute: str, include_frozen: bool = False
    ) -> None:
        """
        Assegna un valore a un attributo dei parametri.

        Args:
            items (Union[List, Dict]): Valori da assegnare (lista o dizionario).
            attribute (str): Nome dell'attributo da assegnare.
            include_frozen (bool, opzionale): Se includere i parametri congelati. Default è False.

        Raises:
            ValueError: Se il numero di elementi non corrisponde al numero di parametri.
            TypeError: Se items non è né una lista né un dizionario.
        """
        if isinstance(items, (list, np.ndarray)):
            params = list(self) if include_frozen else self.free_parameters
            if len(items) != len(params):
                raise ValueError(
                    f"Number of items {len(items)} must match number of {'all' if include_frozen else 'free'} parameters ({len(params)})"
                )
            for param, val in zip(params, items):
                setattr(param, attribute, val)
        elif isinstance(items, dict):
            for name, val in items.items():
                param = self[name]
                if not include_frozen and param.frozen:
                    continue
                setattr(param, attribute, val)
        else:
            raise TypeError("Items must be a list or dictionary")

    def add_parameter(self, parameter: "Parameter", name = None) -> None:
        """
        Aggiunge un parametro al gestore.

        Args:
            parameter (Parameter): Il parametro da aggiungere.

        Raises:
            ValueError: Se il parametro esiste già o se si tenta di aggiungere un parametro dopo la creazione del modello.
        """
        if name is None:
            name = parameter.name

        if self._is_inside_model:
            raise ValueError(
                f"Cannot add parameter {name} to model after it is locked"
            )
        if not isinstance(parameter, Parameter):
            raise TypeError('Addes parameter must be istance of Parameter class')
        
        if name in self._parameters:
            raise ValueError(f"Parameter {name} already exists.")
        
        #if name is None:
        #    name = parameter.name

        self._parameters[name] = parameter
        #self.parameter_map[parameter.name] = parameter.name
        self._invalidate_cache()

    def __getitem__(self, name: str) -> "Parameter":
        """
        Ritorna un parametro usando il suo nome.

        Args:
            name (str): Nome del parametro.

        Returns:
            Parameter: Il parametro richiesto.

        Raises:
            KeyError: Se il parametro non è trovato.
        """
        try:
            return self._parameters[name]
        except KeyError:
            raise KeyError(f"Parameter '{name}' not found.")

    def __setitem__(self, key: str, value: "Parameter") -> None:
        """
        Imposta un parametro usando il suo nome.

        Args:
            key (str): Nome del parametro.
            value (Parameter): Il parametro da impostare.

        Raises:
            ValueError: Se si tenta di impostare un parametro dopo la creazione del modello.
            TypeError: Se value non è un'istanza di Parameter.
            ValueError: Se il parametro esiste già.
        """
        if self._is_inside_model:
            raise ValueError(f"Cannot add parameter {key} to model after it is locked")
        if not isinstance(value, Parameter):
            raise TypeError(
                f"New parameter must be instance of Parameter, not {type(value)}"
            )
        if key in self._parameters:
            raise ValueError(f"Parameter {key} already exists.")
        self._parameters[key] = value
        self._invalidate_cache()

    def __contains__(self, key: str) -> bool:
        """
        Verifica se un parametro è presente usando il suo nome.

        Args:
            key (str): Nome del parametro.

        Returns:
            bool: True se il parametro è presente, False altrimenti.
        """
        return key in self._parameters

    def __iter__(self) -> Iterator["Parameter"]:
        """
        Itera sui parametri.

        Returns:
            Iterator[Parameter]: Iteratore sui parametri.
        """
        return iter(self._parameters.values())

    def __len__(self) -> int:
        """
        Ritorna il numero di parametri.

        Returns:
            int: Numero di parametri.
        """
        return len(self._parameters)
    
    def __str__(self) -> str:
        """
        Ritorna una rappresentazione in formato tabella dei parametri gestiti.

        Returns:
            str: Tabella che mostra nome, valore, bounds e stato frozen dei parametri.
        """
        # Creiamo l'header della tabella
        header = f"{'Name':<20} {'Value':<20} {'Bounds':<30} {'Frozen':<10}\n"
        header += "-" * 80 + "\n"
        
        # Raccogliamo i dati di ogni parametro
        rows = []
        for i, param in enumerate(self):
            name = self.parameters_names[i]
            value = str(param.value) if param.value is not None else "None"
            bounds = str(param.bounds) if param.bounds is not None else "None"
            frozen = "Yes" if param.frozen else "No"
            
            # Aggiungiamo una riga alla tabella
            row = f"{name:<20} {value:<20} {bounds:<30} {frozen:<10}"
            rows.append(row)
        
        # Uniamo l'header con tutte le righe
        table = header + "\n".join(rows)
        return table

    def items(self):
        """
        Ritorna gli elementi del gestore come coppie chiave-valore.

        Returns:
            ItemsView: Vista degli elementi del gestore.
        """
        return self._parameters.items()

    def keys(self):
        """
        Ritorna le chiavi del dizionario base.

        Returns:
            KeysView: Vista delle chiavi del gestore.
        """
        return self._parameters.keys()

    def values(self):
        """
        Ritorna i parametri.

        Returns:
            ValuesView: Vista dei valori del gestore.
        """
        return self._parameters.values()
    
    
    
    
    
    


# Esempi di utilizzo
p1 = Parameter(name="param1", value=5.0, bounds=(0.0, 10.0), frozen=True)
p2 = Parameter(name="param2", value=7.0, bounds=(0.0, 10.0))
p3 = Parameter(name = 'param3', value =3.5)
handler = ParameterHandler([p1, p2])

#handler.add_parameter(p3, name = 'test')
print(handler.parameters_names)
print([p.name for p in handler])



['param1', 'param2']
['param1', 'param2']


In [223]:
print(handler.parameters_names)  # ['param1', 'param2']
print(handler.parameters_values)  # [5.0, 7.0]
print(handler.n_free_params)  # 1
print(
    handler.free_parameters
)  # [<Parameter name='param2', value=7.0, bounds=(0.0, 10.0), frozen=False>]
print(
    handler.frozen_parameters
)  # [<Parameter name='param1', value=5.0, bounds=(0.0, 10.0), frozen=True>]

handler.set_values({"param2": 8.0})
print(handler.parameters_values)  # [5.0, 8.0]

handler.set_bounds({"param2": (0.0, 15.0)})
print(handler.parameters_bounds)  # [(0.0, 10.0), (0.0, 15.0)]

handler.set_frozen({"param1": True})
print(handler.frozen_parameters)  # [<Parameter name='param1', val

print("param1" in handler, len(handler))

handler.set_frozen(is_frozen=[False, False], include_frozen=True)
handler["param1"].value = 2.55  # metodo migliore
print(handler.free_parameters)
print(handler["param1"])
print(handler["param2"])
print(handler)


['param1', 'param2']
[5.0, 7.0]
1
[5.0, 8.0]
[(0.0, 10.0), (0.0, 15.0)]
True 2
[<models.parameters.Parameter object at 0x760f64e13610>, <models.parameters.Parameter object at 0x760f6464d250>]
PARAM NAME: param1
____________________________________________________________________________________________________
NOME            VALORE     FREEZE     BOUNDS               DESCRIZIONE          
____________________________________________________________________________________________________
param1          2.55       0          (0, 10)                                   

PARAM NAME: param2
____________________________________________________________________________________________________
NOME            VALORE     FREEZE     BOUNDS               DESCRIZIONE          
____________________________________________________________________________________________________
param2          8          0          (0, 15)                                   

Name                 Value              

In [224]:
class ParameterMapper:
    '''classe nodo per ParameterHandler, che segue le struttura di CompositeModel in modo da mappare i parametri'''
    def __init__(self, left, right) -> None:
        self._left = left
        self._right = right

        self.param_map = OrderedDict() # (nome, Parameter)
    
    @property
    def left(self):
        return self._left

    @property
    def right(self):
        return self._right
    

In [225]:
def gaussian(x, mu, sigma=1):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def calcola_dimensioni(lista):
    # Variabili di controllo per la sequenza
    dimensioni = 0
    if lista[0] == "x" and lista[1] != "y":
        dimensioni += 1
    elif lista[0] == "x" and lista[1] == "y" and lista[2] != "z":
        dimensioni += 2
    elif lista[0] == "x" and lista[1] == "y" and lista[2] == "z":
        dimensioni += 3

    return dimensioni

def componemodels(op, **kwargs):
    return lambda left, right: CompositeModel(left, right, op, **kwargs)

In [226]:
import inspect
from copy import deepcopy
import warnings

class Model:
    def __init__(
        self, func, parameters, ndim, ninputs, noutputs, name="SimpleModel"
    ) -> None:
        
        self._parameters = parameters
        self._ndim = ndim
        self._ninputs = ninputs
        self._n_outputs = noutputs
        self._callable = func
        self._name = name
        self._grid_variables = None  # Inizializza _grid_variables nel costruttore
        
        #self._update_callable()

    # POINTER PROPRIETA
    @property
    def parameters(self):
        return self._parameters

    @property
    def name(self):
        return self._name
    
    @name.setter
    def name(self, value):
        self._name = value

    @property
    def n_dim(self):
        return self._ndim

    @property
    def n_inputs(self):
        return self._ninputs

    @property
    def n_outputs(self):
        return self._n_outputs

    @property
    def grid_variables(self):
        return self._grid_variables
    
    @property
    def parameters_names(self) -> List[str]:
        return self._parameters.parameters_names

    @property
    def n_parameters(self) -> int:
        return len(self.parameters)

    @property
    def parameters_values(self) -> List[float]:
        return self.parameters.parameters_values

    @property
    def parameters_bounds(self) -> List[Tuple[float, float]]:        
        return self.parameters.parameters_bounds

    @property
    def free_parameters(self) -> List[Parameter]:       
        return self.parameters.free_parameters

    @property
    def frozen_parameters(self) -> List[Parameter]:
        return self.parameters.frozen_parameters

    @property
    def n_free_parameters(self) -> int:
        return self.parameters.n_free_params

    @property
    def _binary_freeze_map(self) -> List[bool]:
        return [p.frozen for p in self]

    @property
    def _binary_melt_map(self) -> List[bool]:
        return [not p.frozen for p in self]
    
    @property
    def left(self):
        return None
    
    @property
    def right(self):
        return None
    
    def set_parameters_values(self, args=None, **kwargs) -> None:
        """
        Imposta i valori dei parametri utilizzando argomenti posizionali o parole chiave.

        Args:
            args (list, opzionale): Una lista di valori per i parametri.
            kwargs (dict, opzionale): Un dizionario con nomi di parametri come chiavi e valori corrispondenti.

        Raises:
            ValueError: Se vengono forniti sia args che kwargs.

        Esempio:
            >>> obj.set_parameters_values([1, 2, 3])
            >>> obj.set_parameters_values(param1=1, param2=2)
        """
        if args:
            self.parameters.set_values(args)
        if kwargs:
            self.parameters.set_values(kwargs)
    
    def set_parameters_bounds(self, args=None, **kwargs) -> None:
        """
        Imposta i limiti dei parametri utilizzando argomenti posizionali o parole chiave.

        Args:
            args (list, opzionale): Una lista di limiti per i parametri.
            kwargs (dict, opzionale): Un dizionario con nomi di parametri come chiavi e limiti corrispondenti.

        Raises:
            ValueError: Se vengono forniti sia args che kwargs.

        Esempio:
            >>> obj.set_parameter_bounds([0, 10])
            >>> obj.set_parameter_bounds(param1=(0, 10), param2=(0, 5))
        """
        if args:
            self.parameters.set_bounds(args)
        if kwargs:
            self.parameters.set_bounds(kwargs)
        
    
    def _set_frozen_state(self, state: bool, *args) -> None:
        """
        Imposta lo stato di congelamento per i parametri specificati o per tutti i parametri.

        Args:
            state (bool): Stato di congelamento (True per congelare, False per scongelare).
            args (tuple): Una lista di nomi o indici dei parametri da congelare/scongelare.

        Esempio:
            >>> obj._set_frozen_state(True, 'param1', 'param2')
            >>> obj._set_frozen_state(False)
        """
        if not args:
            vals = self.parameters_names
        else:
            vals = args
        for element in vals:
            name = element
            if isinstance(element, int):
                name = self.parameters._map_indices_to_names(element)
            self.parameters[name].frozen = state

        self._update_frozen_mask()

    def freeze_parameters(self, *args, **kwargs) -> None:
        """
        Congela i parametri specificati o tutti i parametri se nessuno è specificato.

        Args:
            args (tuple): Una lista di nomi o indici dei parametri da congelare.
            kwargs (dict): Un dizionario con nomi di parametri come chiavi e valori corrispondenti per congelarli a determinati valori.

        Esempio:
            >>> obj.freeze_parameters('param1', 'param2')
            >>> obj.freeze_parameters(param1=1, param2=2)
        """
        if kwargs:
            # posso freezare un parametro ad un determinato valore
            self.set_parameters_values(kwargs)
            args = [*args, *list(kwargs.keys())]
        self._set_frozen_state(True, *args)

    def unfreeze_parameters(self, *args) -> None:
        """
        Scongela i parametri specificati o tutti i parametri se nessuno è specificato.

        Args:
            args (tuple): Una lista di nomi o indici dei parametri da scongelare.

        Esempio:
            >>> obj.unfreeze_parameters('param1', 'param2')
            >>> obj.unfreeze_parameters()
        """
        self._set_frozen_state(False, *args)


    @staticmethod
    def _extract_params(method):
        

        """
        Estrae i nomi e i valori di default dei parametri dal metodo evaluate.

        Parameters:
        -----------
        method : function
            Metodo evaluate della classe.

        Returns:
        --------
        tuple
            Lista dei nomi dei parametri, dei valori di default e dello stato frozen.
        """
        signature = inspect.signature(method)
        params = {}
        is_constant = []
        for param_name, param in signature.parameters.items():
            if param_name != "self":
                if param.default is inspect.Parameter.empty:
                    params[param_name] = 1
                    is_constant.append(False)
                else:
                    params[param_name] = param.default
                    is_constant.append(True)
        return list(params.keys()), list(params.values()), is_constant

    @classmethod
    def from_callable(cls, func, ndim=None, ninputs=1, noutputs=1, name="SimpleModel"):
        names, values, frozen = cls._extract_params(func)

        # check sul numero di dimensioni
        if ndim is None:
            ndim = calcola_dimensioni(names)
            

        parameters = ParameterHandler()

        for i in range(ndim, len(names)):
            parameters.add_parameter(
                Parameter(name=names[i], value=values[i], frozen=frozen[i])
            )
        # func, parameters, ndim, ninputs, noutputs, name="SimpleModel"

        new_cls = cls(func, parameters, ndim, ninputs, noutputs, name)
        new_cls._grid_variables = names[:ndim]  # Sovrascrive _grid_variables
        if ndim == 0:
            new_cls._ndim = 1
        #new_cls._update_callable()
        return new_cls
    
    

    def __str__(self):
        """
        Restituisce una rappresentazione testuale del modello.

        Returns:
            str: Una stringa che rappresenta il modello.
        """
        total_string = f"MODEL NAME: {self.name} \n"
        total_string += f"FREE PARAMS: {self.n_free_parameters}\n"
        total_string += f"GRID VARIABLES: {self.grid_variables}\n"
        total_string +=  f"N-DIM: {self.n_dim}\n"
        total_string += "-" * 60 + "\n"
        total_string += (
            f"{'':<4} {'NAME':<15} {'VALUE':<10} {'IS-FROZEN':<10} {'BOUNDS':<20}\n"
        )
        total_string += "-" * 60 + "\n"
        for i, param in enumerate(self._parameters):
            value_str = f"{param.value:.2f}"
            bounds_str = f"({param.bounds[0]:.2f}, {param.bounds[1]:.2f})"
            frz = "Yes" if param.frozen else "No"
            total_string += (
                f"{i:<4} {param.name:<15} {value_str:<10} {frz:<10} {bounds_str:<20}\n"
            )
        return total_string
    
    def __call__(self, grid = [], *args, **kwargs):
        if not args and not kwargs:
            return self._callable(*grid,*self.parameters_values)
        
        ## check lenti ma necessari
        vals = self.parameters_values
        if args:
            if len(args) > len(self.parameters_names):
                raise ValueError('Number of inputs > number of parameters')
            if len(args) > len(self._not_frozen_indeces):
                warnings.warn(f'parameters {[p.name for p in self.frozen_parameters]} are frozen, theyr args will be ignored!')
            for i,idx in enumerate(self._not_frozen_indeces):
                vals[idx] =  args[i]
                
        if kwargs:
            for key in kwargs.keys():
                if self.parameters[key].frozen is True:
                    warnings.warn(f'Parameter {key} is frozen, kwargs will be ignored!')
            for i,idx in enumerate(self._not_frozen_indeces):
                if self.parameters[self.parameters_names[idx]].frozen is False:
                    vals[idx] = kwargs[self.parameters_names[idx]]

                    

        return self._callable(*vals)

    def _update_frozen_mask(self):
        self._not_frozen_indeces = [i for i in range(len(self._binary_freeze_map)) if self._binary_freeze_map[i] is False]
        self._frozen_indeces = [
            i for i in self._binary_freeze_map if self._binary_freeze_map[i] is True
        ]
            
            

        
    def __getitem__(self, name: str) -> Parameter:
        return self.parameters[name]

    def __setitem__(self, key, value: Parameter) -> None:
        return self.parameters.__setitem__(key, value)

    def __contains__(self, key: str) -> bool:
        return self.parameters.__contains__(key)

    def __iter__(self):
        return self.parameters.__iter__()

    def __len__(self) -> int:
        return self.parameters.__len__()
    
    def copy(self):
        return deepcopy(self)
    
    def evaluate(self, *args, **kwargs):
        return self._callable(*args, **kwargs)
    
    __add__ = componemodels("+")
    __mul__ = componemodels("*")
    __or__ = componemodels("|")
    __truediv__ = componemodels("/")
    __sub__ = componemodels("-")

    

In [227]:
model1 = Model.from_callable(gaussian, name='Model1')
print(model1)

model2 = Model.from_callable(gaussian, name = 'Model2')
print(model2)

MODEL NAME: Model1 
FREE PARAMS: 1
GRID VARIABLES: ['x']
N-DIM: 1
------------------------------------------------------------
     NAME            VALUE      IS-FROZEN  BOUNDS              
------------------------------------------------------------
0    mu              1.00       No         (-inf, inf)         
1    sigma           1.00       Yes        (-inf, inf)         

MODEL NAME: Model2 
FREE PARAMS: 1
GRID VARIABLES: ['x']
N-DIM: 1
------------------------------------------------------------
     NAME            VALUE      IS-FROZEN  BOUNDS              
------------------------------------------------------------
0    mu              1.00       No         (-inf, inf)         
1    sigma           1.00       Yes        (-inf, inf)         



In [237]:
import operator

class CompositeModel(Model):

    LINEAR_OPERATIONS = ["+", "-", "*", "/", "**"]
    COMPOSITE_OPERATION = "|"

    IS_COMPOSITE = True
    _parameters = OrderedDict()
    _name = 'CompositeModel'

    def __init_subclass__(cls) -> None:
        return super().__init_subclass__()

    def __init__(self, left = None, right = None, op = '+') -> None:
        self._left = left
        self._right = right
        self.op_str = op
        self._operator = self.map_operator(op)
        self._parameters = self._init_parameters()
        self._n_dim, self._n_inputs, self._n_outputs = self._update_n_dim()
        #self.name = 'CompositeModel'
        self.submodels = self._collect_submodels()
    
    @property
    def left(self):
        return self._left
    
    @property
    def right(self):
        return self._right
    
    @property
    def parameters(self):
        return self._parameters
    
    @property
    def n_dim(self):
        return self._n_dim

    @property
    def n_inputs(self):
        return self._n_inputs

    @property
    def n_outputs(self):
        return self._n_outputs
    
    @property
    def grid_variables(self):
        return self.left.grid_variables
    
    
    def map_operator(self, op):
        """
        Mappa l'operatore dato a una funzione corrispondente.

        Args:
            op (str): Operatore come stringa.

        Returns:
            function: Funzione corrispondente all'operatore.

        """
        if op == "+":
            val = operator.add
        elif op == "/":
            val = operator.truediv
        elif op == "*":
            val = operator.mul
        elif op == "-":
            val = operator.sub
        else:
            val = None
        return val
    
    def _update_n_dim(self):
        """
        Controlla gli inputs e gli outputs dei sottomodelli per essere sicuro
        che le operazioni binarie siano supportate.
        corregge il numero di inputs/outputs del modello composito di conseguenza
        """

        if self.op_str in self.LINEAR_OPERATIONS:
            if (self.left.n_inputs != self.right.n_inputs) or (
                self.left.n_outputs != self.right.n_outputs
            ):
                raise ValueError("Number of inputs/output do not match!")

            if self.left.n_dim != self.right.n_dim:
                raise ValueError("Number of dimensions do not match!")

            n_dim = self.left.n_dim
            n_inputs = self.left.n_inputs
            n_outputs = self.left.n_outputs

        elif self.op_str == self.COMPOSITE_OPERATION:
            if self.left.n_outputs != self.right.n_inputs:
                raise ValueError(
                    "Number of output for left must be = n_inputs of right!"
                )
            if self.left.n_dim != self.right.n_dim:
                raise ValueError("Number of dimensions do not match!")

            n_dim = self.left.n_dim
            n_inputs = self.left.n_inputs
            n_outputs = self.right.n_outputs
        
        return  n_dim, n_inputs, n_outputs
        
    def _collect_submodels(self):
        leaf = []   #lista nomi dei modelli
        param_map = OrderedDict()
        submodels = []
        
        
        def dfs(node):
            nonlocal leaf, param_map, submodels
            if not node:
                return
            
            if not node.left and not node.right:
                leaf.append(node)
                submodels.append(node.name)
                return           
            
            dfs(node.left)            
            dfs(node.right)
        
        dfs(self)

        # costruzione della mappa dei parametri
        #n = 0
        #for model in leaf:
        #    for param in model:
        #        param_map[param.name + '_'+str(n)] = param
        #    n += 1

        return submodels
    
    def composite_structure(self):
        """
        Restituisce una stringa che rappresenta la logica con cui i sottomodelli sono uniti.

        Returns:
            str: Una stringa che rappresenta la struttura dell'albero binario del modello composito.
        """

        def helper(m, id_counter):
            if isinstance(m, CompositeModel):
                left_str, id_counter = helper(m.left, id_counter)
                right_str, id_counter = helper(m.right, id_counter)
                return f"({left_str} {m.op_str} {right_str})", id_counter
            else:
                return f"{m.name} [{id_counter}]", id_counter + 1

        structure, _ = helper(self, 0)
        return structure
    
    def _init_parameters(self):
        '''Crea un Nuovo ParameterHandler con i nomi cambiati dei parametri ma mappati
        agli stessi parametri originali
        '''

        parameters = ParameterHandler()
        n = 0

        def dfs(node):
            nonlocal n, parameters
            if node is None:
                return
            
            if not node.left and not node.right:
                for param in node:
                    name = param.name + f'_{n}'
                    parameters.add_parameter(param, name=name)
                n += 1
                
            
            dfs(node.left)
            dfs(node.right)
        
        dfs(self)
            
        return parameters
    
    def __str__(self):
        """
        Restituisce una stringa che rappresenta il modello composito e i suoi parametri.

        Returns:
            str: Una stringa che rappresenta il modello composito, i modelli contenuti e i parametri liberi.
        """
        total_string = f"COMPOSITE MODEL NAME: {self.name} \n"
        total_string += (
            f"CONTAINED MODELS: {self.submodels}\n"
        )
        total_string += (
            f"GRID VARIABLES: {self.grid_variables}\n"
        )
        total_string += f"LOGIC: {self.composite_structure()}\n"
        total_string += f"FREE PARAMS: {self.n_free_parameters}\n"
        total_string += "-" * 60 + "\n"
        total_string += (
            f"{'':<4} {'NOME':<15} {'VALORE':<10} {'FREEZE':<10} {'BOUNDS':<20} \n"
        )
        total_string += "-" * 60 + "\n"
        for i, (param_name, param) in enumerate(self.parameters.items()):
            value_str = f"{param.value:.2f}"
            bounds_str = f"({param.bounds[0]:.2f}, {param.bounds[1]:.2f})"
            frz = 'Yes' if param.frozen else 'No'
            total_string += f"{i:<4} {param_name:<15} {value_str:<10} {frz:<10} {bounds_str:<20}\n"
        return total_string

    def _internal_call(self,*args, **kwargs):
        return
    
    def __call__(self):
        return


        


In [235]:
print(model1.n_dim, model1.n_outputs)
print(model2.n_dim, model2.n_outputs)



1 1
1 1
[1, 2]


In [238]:
Cmodel = model1 + model2 

#Cmodel.freeze_parameters('mu_1')
print(Cmodel.evaluate())


[1, 1] [1, 1]
0.7978845608028654


In [214]:
'''def func(a, b, c):
    return  a+b+c


model = Model.from_callable(func, name='Test')
model.freeze_parameters("b")
print(model)
print(model._binary_freeze_map)
print(model.n_dim, model.n_inputs, model.n_outputs)
model(1,4)  # '2' is intended for 'c' but gets assigned to 'c' due to 'b' being frozen

model2 = model.copy()
## test per vedere se  arrivo allo stesso parametro
## confermo copia modello crea nuovi parametri
print(model.parameters._parameters)
print(model2.parameters._parameters)
'''

'def func(a, b, c):\n    return  a+b+c\n\n\nmodel = Model.from_callable(func, name=\'Test\')\nmodel.freeze_parameters("b")\nprint(model)\nprint(model._binary_freeze_map)\nprint(model.n_dim, model.n_inputs, model.n_outputs)\nmodel(1,4)  # \'2\' is intended for \'c\' but gets assigned to \'c\' due to \'b\' being frozen\n\nmodel2 = model.copy()\n## test per vedere se  arrivo allo stesso parametro\n## confermo copia modello crea nuovi parametri\nprint(model.parameters._parameters)\nprint(model2.parameters._parameters)\n'

In [215]:
'''def simple_test(a,b,c):
    return a +b +c

test = Model.from_callable(simple_test)

test.freeze_parameters(b = 2)
test.set_parameters_values(a = 10)

print(test)
print(test())
print(test.evaluate(1,1, 1))'''

'def simple_test(a,b,c):\n    return a +b +c\n\ntest = Model.from_callable(simple_test)\n\ntest.freeze_parameters(b = 2)\ntest.set_parameters_values(a = 10)\n\nprint(test)\nprint(test())\nprint(test.evaluate(1,1, 1))'